# ARC26 shared-frontier bounded four-task smoke

Production-faithful public-validation gate for the final shared-view, cheap-first, frontier-resume, bounded-selector submission path.


In [ ]:
import os

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["OMP_NUM_THREADS"] = "12"


In [ ]:
MODE = "validation"

CODE_DATASET_ROOT = "/kaggle/input/datasets/yuvraj/arc2026"
MODEL_PATH = "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
COMP_ROOT = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"

VALIDATION_KEYS = ['5dbc8537', 'b6f77b65', '142ca369', '38007db0']
NPROCS = 4
DFS_PROB_THRESHOLD = 0.2
UNSLOTH_MULTITOKEN_REPEAT_LEN = 9
EVAL_COLOR_PERMUTATIONS = 3
PROFILE_TIMINGS = True

ADAPTIVE_DFS_PROB_THRESHOLD = 0.1
ADAPTIVE_COLOR_PERMUTATIONS = 3
ADAPTIVE_MIN_UNIQUE_CANDIDATES = 2

VALIDATION_END_TIME_HOURS = 1.0
SUBMIT_COMPETITION_END_TIME_HOURS = 11 + 50 / 60
RESET_RUN_ARTIFACTS = True

WORK_NOTEBOOK_ROOT = "/kaggle/working/arc2026_shared_frontier_smoke4"
WORK_CODE_DIR = WORK_NOTEBOOK_ROOT + "/ARC-AGI1/qwen_baseline"
WRITABLE_UNSLOTH_PARENT = "/kaggle/working/shared_frontier_smoke4_stack"


In [ ]:
import os
from pathlib import Path


def _truthy_env(name: str) -> bool:
    value = os.getenv(name)
    if value is None:
        return False
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


IS_KAGGLE_RERUN = _truthy_env("KAGGLE_IS_COMPETITION_RERUN")
assert MODE in {"validation", "submit_competition"}
EFFECTIVE_MODE = "submit_competition" if IS_KAGGLE_RERUN else MODE

EVAL_CHALLENGES = f"{COMP_ROOT}/arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS = f"{COMP_ROOT}/arc-agi_evaluation_solutions.json"
TEST_CHALLENGES = f"{COMP_ROOT}/arc-agi_test_challenges.json"

if IS_KAGGLE_RERUN:
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR = "/kaggle/working/inference_outputs_vanilla_v2_q9_24_submit"
    SUBMISSION_PATH = "/kaggle/working/submission.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = None
    END_TIME_HOURS = SUBMIT_COMPETITION_END_TIME_HOURS
    RUN_INFERENCE = True
elif MODE == "validation":
    TEST_PATH = EVAL_CHALLENGES
    SOLUTION_PATH = EVAL_SOLUTIONS
    OUTPUT_DIR = "/kaggle/working/inference_outputs_vanilla_v2_q9_24_validation"
    SUBMISSION_PATH = "/kaggle/working/validation_submission_unsloth_q9.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = VALIDATION_KEYS
    END_TIME_HOURS = VALIDATION_END_TIME_HOURS
    RUN_INFERENCE = True
else:
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR = "/kaggle/working/inference_outputs_vanilla_v2_q9_24_shortcut"
    SUBMISSION_PATH = "/kaggle/working/submission.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = None
    END_TIME_HOURS = 0.0
    RUN_INFERENCE = False

ADAPTIVE_OUTPUT_DIR = "/kaggle/working/shared_frontier_smoke4_adaptive"

print("mode_requested =", MODE)
print("is_kaggle_rerun =", IS_KAGGLE_RERUN)
print("effective_mode =", EFFECTIVE_MODE)
print("test_path =", TEST_PATH)
print("output_dir =", OUTPUT_DIR)
print("submission_path =", SUBMISSION_PATH)
print("selected_keys =", SELECTED_KEYS)
print("end_time_hours =", END_TIME_HOURS)
print("run_inference =", RUN_INFERENCE)


In [ ]:
import importlib.util
import os
import shutil
import sys
from pathlib import Path

assert Path(CODE_DATASET_ROOT).exists(), f"Missing code dataset root: {CODE_DATASET_ROOT}"
assert Path(MODEL_PATH).exists(), f"Missing model path: {MODEL_PATH}"
assert Path(TEST_PATH).exists(), f"Missing challenge path: {TEST_PATH}"
assert Path(os.environ["TRITON_PTXAS_PATH"]).exists(), os.environ["TRITON_PTXAS_PATH"]
if SOLUTION_PATH is not None:
    assert Path(SOLUTION_PATH).exists(), f"Missing solution path: {SOLUTION_PATH}"

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"

os.chdir("/kaggle/working")
print("setup cwd =", os.getcwd())

if RESET_RUN_ARTIFACTS:
    for path in [WORK_NOTEBOOK_ROOT, OUTPUT_DIR, WRITABLE_UNSLOTH_PARENT]:
        shutil.rmtree(path, ignore_errors=True)
    try:
        Path(SUBMISSION_PATH).unlink()
    except FileNotFoundError:
        pass
    for path in Path("/kaggle/working").glob("worker_train_*"):
        if path.is_file():
            path.unlink()

for module_name in ["unsloth", "transformers", "torch"]:
    spec = importlib.util.find_spec(module_name)
    print(module_name, spec.origin if spec else "MISSING")


In [ ]:
import os
import shutil
from pathlib import Path

src = Path(CODE_DATASET_ROOT)
dst = Path(WORK_NOTEBOOK_ROOT)
shutil.copytree(src, dst)

required_files = [
    "starter.py",
    "arc_solver.py",
    "arc_search_multitoken.py",
    "patch_unsloth_qwen3_multitoken.py",
]
for name in required_files:
    assert Path(WORK_CODE_DIR, name).is_file(), (
        f"arc2026 is stale: missing {name}"
    )

starter_source = Path(WORK_CODE_DIR, "starter.py").read_text()
solver_source = Path(WORK_CODE_DIR, "arc_solver.py").read_text()
assert "--use-unsloth-multitoken-dfs" in starter_source
assert "--eval-color-permutations" in starter_source
assert "UNSLOTH_COMPILE_LOCATION" in starter_source, "arc2026 starter.py lacks the worker import-race guard"
assert '"embed_tokens"' in solver_source and '"lm_head"' in solver_source
assert "inference_turbo_dfs_multitoken" in solver_source
print("arc2026 multi-token production preflight passed")
print("work_code_dir =", WORK_CODE_DIR)


In [ ]:
spec = importlib.util.find_spec("unsloth")
assert spec is not None and spec.submodule_search_locations
mounted_unsloth = Path(next(iter(spec.submodule_search_locations)))
qwen_source = (mounted_unsloth / "models" / "qwen3.py").read_text()
assert "A = flash_attn_func(Qnn, Knn, Vnn)" in qwen_source

writable_parent = Path(WRITABLE_UNSLOTH_PARENT)
writable_unsloth = writable_parent / "unsloth"
shutil.copytree(mounted_unsloth, writable_unsloth)

sys.path.insert(0, WORK_CODE_DIR)
from patch_unsloth_qwen3_multitoken import PATCH_MARKER, patch_unsloth

changed = patch_unsloth(writable_unsloth)
assert PATCH_MARKER in (writable_unsloth / "models" / "qwen3.py").read_text()
print("writable_unsloth =", writable_unsloth)
print("patched =", [str(path) for path in changed])

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = str(writable_parent) + os.pathsep + RUN_ENV.get("PYTHONPATH", "")


In [ ]:
import json
import os
import subprocess
import sys
import time

if RUN_INFERENCE:
    cmd = [
        sys.executable,
        "starter.py",
        "--test-path", TEST_PATH,
        "--model-path", MODEL_PATH,
        "--output-dir", OUTPUT_DIR,
        "--nprocs", str(NPROCS),
        "--use-unsloth-multitoken-dfs",
        "--unsloth-multitoken-repeat-len", str(UNSLOTH_MULTITOKEN_REPEAT_LEN),
        "--dfs-prob-threshold", str(DFS_PROB_THRESHOLD),
        "--eval-color-permutations", str(EVAL_COLOR_PERMUTATIONS),
        "--shared-eval-augmentations",
        "--adaptive-output-dir", ADAPTIVE_OUTPUT_DIR,
        "--adaptive-dfs-prob-threshold", str(ADAPTIVE_DFS_PROB_THRESHOLD),
        "--adaptive-color-permutations", str(ADAPTIVE_COLOR_PERMUTATIONS),
        "--adaptive-min-unique-candidates", str(ADAPTIVE_MIN_UNIQUE_CANDIDATES),
        "--cheap-first",
        "--end-time", str(time.time() + END_TIME_HOURS * 3600),
    ]
    if PROFILE_TIMINGS:
        cmd.append("--profile-timings")
    if SELECTED_KEYS is not None:
        cmd.extend(["--keys-json", json.dumps(SELECTED_KEYS)])

    print("running:", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=WORK_CODE_DIR, env=RUN_ENV, check=True)
else:
    print("save-version shortcut: full inference runs only during the competition rerun")


In [ ]:
import json
import shutil
import sys
from pathlib import Path

import numpy as np

if WORK_CODE_DIR not in sys.path:
    sys.path.insert(0, WORK_CODE_DIR)

from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_kgmon
from arc_shared_selection import select_bounded_shared_support, fill_missing_attempts

data = ArcDataset.from_file(TEST_PATH, keys=SELECTED_KEYS).load_replies(SOLUTION_PATH)
split_data = data.split_multi_replies()
Path(ADAPTIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

primary_ranked, primary_counts, primary_decoder = select_bounded_shared_support(
    data, split_data, OUTPUT_DIR, support_weight=2.0, n_guesses=2
)
primary_kgmon = primary_decoder.run_selection_algo(score_kgmon)

combined_decoder = ArcDecoder(split_data, n_guesses=2)
if Path(OUTPUT_DIR).exists():
    combined_decoder.load_decoded_results(OUTPUT_DIR)
if Path(ADAPTIVE_OUTPUT_DIR).exists():
    combined_decoder.load_decoded_results(ADAPTIVE_OUTPUT_DIR, run_name=".adaptive")
combined_ranked = combined_decoder.run_selection_algo(score_kgmon)

safe_adaptive = fill_missing_attempts(
    primary_ranked, combined_ranked, primary_counts, n_guesses=2
)

def score(selected):
    submission = data.get_submission(selected)
    return data.validate_submission(submission)

def oracle(decoder):
    total = 0.0
    for base_key, records in decoder.decoded_results.items():
        puzzle_key, _ = base_key.rsplit("_", 1)
        gold = split_data.replies[base_key][0]
        if any(np.array_equal(record["solution"], gold) for record in records.values()):
            total += 1 / len(data.queries[puzzle_key]["test"])
    return total

summary = {
    "requested_tasks": len(SELECTED_KEYS),
    "requested_outputs": len(split_data.keys),
    "primary_decoded_outputs": len(primary_decoder.decoded_results),
    "adaptive_decoded_outputs": len(
        {
            path.name.split(".")[0]
            for path in Path(ADAPTIVE_OUTPUT_DIR).glob("*")
            if path.is_file()
        }
    ) if Path(ADAPTIVE_OUTPUT_DIR).exists() else 0,
    "primary_zero_candidate_outputs": sum(value == 0 for value in primary_counts.values()),
    "primary_one_candidate_outputs": sum(value == 1 for value in primary_counts.values()),
    "primary_two_plus_candidate_outputs": sum(value >= 2 for value in primary_counts.values()),
    "primary_kgmon_score": score(primary_kgmon),
    "primary_bounded_support_score": score(primary_ranked),
    "safe_adaptive_score": score(safe_adaptive),
    "primary_oracle": oracle(primary_decoder),
    "combined_oracle": oracle(combined_decoder),
    "primary_unique_counts": primary_counts,
}

SUMMARY_PATH = Path("/kaggle/working/shared_adaptive49_summary.json")
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
shutil.make_archive("/kaggle/working/shared_primary_candidates", "zip", OUTPUT_DIR)
shutil.make_archive("/kaggle/working/shared_adaptive_candidates", "zip", ADAPTIVE_OUTPUT_DIR)
print(json.dumps(summary, indent=2, sort_keys=True))
